# AI Engineering Interview Prep
## Section: AI Infrastructure & Scalability

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


---
## 🖥️ Section 12 — AI Infrastructure and Scalability

### Q1. How do you improve inference speed in production LLM deployments?
**A:** Key optimisations in order of impact:

1. **Continuous batching (vLLM):** Instead of waiting for a batch to complete, new requests join the running batch at the token level. Maximises GPU utilisation. 3-10× throughput improvement.

2. **Speculative decoding:** Small draft model generates k tokens speculatively; large target model verifies in one pass. Net speedup: 2-3× for the target model.

3. **KV cache optimisation (Paged Attention):** Prevents memory fragmentation, allows more concurrent requests.

4. **Flash Attention 2/3:** Optimised attention kernel with 2-4× speedup, O(N) memory.

5. **Quantisation (INT8/INT4):** Reduce model precision → smaller model → less memory → more requests per GPU.

6. **Tensor parallelism:** Split model across multiple GPUs → lower per-GPU memory → larger batch sizes.

7. **Prefix caching:** Cache KV for common prefixes (system prompts) → avoid recomputation.

🏷️ *Groq uses custom LPDDR-based silicon rather than GPU — achieves 400+ tokens/sec on Llama 3.3-70b by eliminating GPU memory bottleneck.*


### Q2. LLM optimisation techniques
**A:** Comprehensive list of LLM optimisation approaches:

**Model-level:**
- Quantisation (INT8, INT4, FP16, NF4)
- Pruning (remove unimportant weights)
- Knowledge distillation (smaller student model)
- Architecture improvements (GQA, MQA, SlidingWindowAttention)

**Inference-level:**
- Continuous batching (vLLM)
- Speculative decoding (draft + verify)
- KV cache (avoid recomputation)
- Paged Attention (efficient KV memory)
- Flash Attention (fused attention kernel)
- Prefix caching (cache common prompt prefixes)

**System-level:**
- Model parallelism (tensor, pipeline, data)
- GPU-CPU offloading (load balance between GPU and CPU memory)
- Request batching and priority queuing
- Model sharding for 70B+ models

**Compilation:**
- torch.compile() — graph-level optimisations
- TensorRT-LLM — NVIDIA-optimised inference kernels
- GGUF + llama.cpp — CPU-optimised inference

**Hardware:**
- Groq (LPDDR-based, low latency)
- H100 (4× faster than A100 for transformer workloads)
- Apple Silicon (efficient for small models locally)


### Q3. How do you select GPUs for LLM inference?
**A:**
**Key metrics:**
- **Memory bandwidth** (most important for inference): how fast can weights be loaded from GPU memory to compute cores?
- **VRAM capacity:** 7B FP16 model needs 14GB minimum; 70B FP16 needs 140GB
- **Tensor core throughput:** for compute-bound operations (prefill of long prompts)

**GPU comparison:**
| GPU | VRAM | Bandwidth | Suited for |
|-----|------|-----------|------------|
| RTX 4090 | 24GB | 1,008 GB/s | 7B-13B FP16; QLoRA fine-tuning |
| A10G | 24GB | 600 GB/s | Cloud production 7B-13B |
| A100 80GB | 80GB | 2,039 GB/s | 70B models; high-throughput serving |
| H100 80GB | 80GB | 3,350 GB/s | Frontier models; maximum throughput |
| 8×A100 | 640GB | 16 TB/s | LLaMA 405B, GPT-4 class models |

**For self-hosting:** A100 80GB is the production sweet spot (price/performance). H100 for maximum throughput.

**Rule of thumb:** VRAM = model_params × bytes_per_param × 1.2 (overhead). For 7B INT4: 7B × 0.5B × 1.2 = 4.2GB minimum.


### Q4. What is model parallelism vs data parallelism in distributed training?
**A:**
**Data Parallelism (DP):**
- Same model replicated on N GPUs
- Each GPU processes a different subset of the training data
- Gradients aggregated (averaged) across GPUs after each step
- Works when model fits on one GPU
- Linear speedup with number of GPUs

**Model Parallelism (MP):**
- Model is too large for one GPU → split across multiple GPUs
- Types:
  - **Tensor parallelism:** split individual weight matrices across GPUs (Megatron-LM)
  - **Pipeline parallelism:** different layers on different GPUs; sequential execution
  - **Sequence parallelism:** split the sequence dimension across GPUs (for very long sequences)
- Required for 70B+ models

**Hybrid:**
- Most large model training uses data + tensor + pipeline parallelism together
- GPT-3 training: 3D parallelism (data × tensor × pipeline)
- ZeRO (DeepSpeed): data parallel but shards optimizer states, gradients, and weights across GPUs


### Q5. What is tensor parallelism, and how does it help serve large models?
**A:** Tensor parallelism (TP) splits individual weight matrices across multiple GPUs. Named after the Megatron-LM approach.

**For an MLP layer:**
- Weight matrix W (shape d×4d): split column-wise across N GPUs → each GPU holds W columns of shape d×(4d/N)
- Each GPU computes its partial output; results are summed (AllReduce) to get the full output
- Each layer requires one AllReduce communication per GPU

**For self-attention:**
- Attention heads split across GPUs: N GPUs, each handles H/N attention heads
- Q, K, V projections split accordingly

**Benefits:**
- 70B model: with TP=8, each GPU only needs 70B/8 ≈ 9B params × 2 bytes = ~18GB → fits on A100
- Reduces per-GPU memory proportionally to TP degree
- Enables serving very large models that can't fit on one GPU

**Limitation:** requires high-bandwidth GPU interconnect (NVLink for intra-node, InfiniBand for inter-node) — communication overhead is significant.


### Q6. What is pipeline parallelism?
**A:** Pipeline parallelism splits the model's layers across GPUs, with data flowing sequentially from one GPU to the next.

```
GPU 1: Layers 1-8  (embedding + first 8 transformer blocks)
GPU 2: Layers 9-16 (next 8 transformer blocks)
GPU 3: Layers 17-24 (next 8 transformer blocks)
GPU 4: Layers 25-32 (last 8 blocks + head)
```

**Micro-batching:** Split a batch into micro-batches so multiple micro-batches can fill the pipeline simultaneously — reduces pipeline "bubble" idle time.

**Benefits:** Simpler communication than tensor parallelism (only pass activations between adjacent stages); works over slower interconnects.

**Limitations:**
- Pipeline bubble: GPUs are idle during the first fill and last drain of the pipeline
- Higher activation memory per stage (need to store activations for the backward pass)
- Requires careful load balancing across stages

**In practice:** Used in combination with tensor parallelism and data parallelism for very large model training.


### Q7. How does continuous batching improve LLM inference throughput?
**A:** Traditional (static) batching: wait for N requests → process batch → return all at once. Problem: a long request holds up all short requests in the batch.

**Continuous batching (iteration-level batching / in-flight batching):**
- Process requests one token at a time
- After generating each token, check if any request has finished (EOS token)
- If finished, remove it from the batch and immediately add a new waiting request
- The batch size is maintained by swapping in new requests as old ones complete

**Impact:**
- GPU utilisation goes from ~40-50% (static batching with variable-length requests) to ~80-90%
- Throughput: 3-10× improvement over static batching
- Latency: shorter requests don't wait for long requests to finish

**Implemented in:** vLLM (the primary reason vLLM is the production standard), TensorRT-LLM, Hugging Face TGI.

🏷️ *When I deploy Nyaya-Pro's LLM component at scale, I use vLLM for exactly this reason — variable-length legal queries benefit enormously from continuous batching.*


### Q8. What is speculative decoding, and how does it speed up inference?
**A:** LLM inference is memory-bandwidth-bound — the bottleneck is loading model weights from GPU memory, not computing. Speculative decoding exploits this.

**Algorithm:**
1. A small "draft" model (fast) generates k tokens speculatively
2. The large "target" model (slow) verifies all k tokens in ONE forward pass (parallel verification is as cheap as generating 1 token)
3. Accept all tokens up to the first mismatch; reject the rest
4. Resume draft model from the last accepted token

**Why it works:** Loading large model weights for 1 token ≡ loading for k tokens (same bandwidth cost). Verification of k tokens costs ≈ 1 generation step for the target model.

**Speed gain:** 2-3× faster target model effective throughput. Typical: draft model generates k=5 tokens; target accepts 3-4 on average → ~3.5× speedup.

**Models:** Medusa (multiple draft heads on the same model), Lookahead decoding, Eagle (speculative decoding with tree-based drafts).


### Q9. What is KV cache, and how do you manage memory for it?
**A:** See LLM Fundamentals Q27 for how KV cache works. Here: memory management.

**Memory consumption:**
```
KV cache size = num_layers × 2 × num_heads × head_dim × seq_len × batch_size × bytes_per_element
```
For Llama 3 70B (80 layers, 8 KV heads, 128 head_dim, FP16):
- 1 request, 4096 tokens: 80 × 2 × 8 × 128 × 4096 × 2 bytes = 5.4GB
- 10 concurrent requests: 54GB just for KV cache!

**Management strategies:**
1. **Paged Attention (vLLM):** KV cache in non-contiguous 16-token "pages" (like OS virtual memory). No pre-allocation waste; efficient sharing for parallel beams.
2. **Sliding window attention (Mistral):** Only keep last W=4096 tokens' KV; bounded cache size regardless of sequence length.
3. **KV quantisation (INT8):** Store K,V in INT8 → 2× memory savings with ~1% quality loss.
4. **GQA/MQA:** Fewer K,V heads → proportionally less cache.
5. **Preemption:** vLLM can preempt low-priority requests' KV cache to serve high-priority ones.


### Q10. What is Paged Attention?
**A:** Paged Attention (vLLM, 2023) manages KV cache like an operating system manages virtual memory — using "pages" of fixed size.

**Problem with standard KV cache:** pre-allocate the maximum sequence length for every request upfront → enormous waste (most requests are much shorter than max). Also, can't share KV cache between requests that share a prefix.

**Paged Attention solution:**
1. KV cache is divided into "blocks" (pages) of size 16 tokens
2. Each request gets blocks allocated on-demand as needed
3. Non-contiguous blocks are mapped to physical GPU memory via a block table
4. When a request completes, blocks are freed immediately
5. Requests sharing a common prefix (same system prompt) can **share the same physical blocks** → massive memory savings for multi-tenant serving

**Impact:** vLLM reported 24× higher throughput compared to HuggingFace Transformers in the original paper. Now the standard approach for production LLM serving.


### Q11. How does GGUF work?
**A:** GGUF (GPT-Generated Unified Format) is a binary file format for storing quantised LLM model weights, designed for efficient CPU inference with llama.cpp.

**History:** GGUF replaced the older GGML format (2023). Designed to be extensible — can store model hyperparameters, tokeniser vocabulary, metadata, and quantised weights all in one file.

**Quantisation types supported:**
- Q4_K_M, Q4_K_S: 4-bit quantisation with k-means (M=medium quality, S=small/fast)
- Q5_K_M, Q5_K_S: 5-bit (higher quality)
- Q8_0: 8-bit (near-full quality)
- IQ4_XS: "importance-aware quantisation" — quantise important layers at higher precision

**Why GGUF matters:**
- Run 7B models on 8GB RAM CPU
- Run 13B models on 16GB RAM CPU
- Run 70B on Mac Studio (96GB unified memory)
- No GPU required

**Workflow:** Download GGUF from HuggingFace → run with llama.cpp or Ollama → fully local, private, offline.


### Q12. How do you optimise inference for edge and mobile deployment?
**A:**
**Target constraints:** battery, limited RAM (4-8GB), CPU only or mobile NPU, network-optional, privacy.

**Optimisation stack:**
1. **Model selection:** use SLMs (Phi-3 Mini 3.8B, Llama 3 8B, Gemma 2B)
2. **Quantisation:** 4-bit (GGUF Q4_K_M) — fits 7B in 4-5GB RAM
3. **Inference runtime:**
   - iOS/macOS: Core ML (Apple Neural Engine), MLX framework
   - Android: MediaPipe LLM Inference API, ONNX Runtime
   - Cross-platform: llama.cpp, MLC-LLM (compiles to WebGPU, Metal, Vulkan)
4. **Speculative decoding:** draft model on CPU, verify with target on NPU
5. **Caching:** cache the system prompt's KV to avoid re-processing on each query
6. **Lazy loading:** load model layers on demand from disk if RAM is very constrained

**Practical targets:**
- iPhone 15 Pro: ~20 tokens/sec with Llama 3 8B Q4 via Core ML
- Raspberry Pi 4: ~5 tokens/sec with Llama 3 2B Q4 via llama.cpp


### Q13-27: Infrastructure Topics (Concise)

### Q13. Model quantisation types (INT8, INT4, FP16, BF16) and quality effects
- **BF16:** same range as FP32, 16-bit. Standard for GPU training. Negligible quality loss.
- **FP16:** slightly narrower range than BF16. Also near-zero quality loss for inference.
- **INT8:** 8-bit integer. 2× smaller than FP16. ~1% quality loss. Standard for deployment.
- **INT4/NF4:** 4-bit. 4× smaller. ~2-3% quality loss. Used in QLoRA, GPTQ, AWQ. Good for consumer GPU.
- **Binary (1-bit):** 32× smaller. Significant quality loss. Emerging (BitNet b1.58).

---

### Q14. How do you implement auto-scaling for AI workloads?
Use Kubernetes HPA (Horizontal Pod Autoscaler) based on custom metrics: GPU utilisation, queue depth, TTFT latency. Scale out when GPU util > 80% or queue > 50 requests. Scale in after 10 minutes below 30% utilisation. Use Karpenter or AWS EKS Auto for GPU node provisioning.

---

### Q15. Role of load balancing in AI serving infrastructure
Route requests across inference server pool to: even GPU utilisation, route to nearest healthy server, sticky sessions for KV cache sharing (same system prompt → same server). Use least-loaded routing (not simple round-robin) to account for variable request lengths.

---

### Q16. How do you manage GPU memory for serving multiple models?
MPS (Multi-Process Service) for sharing one GPU across models. Model multiplex: load/unload models on demand with LRU eviction. NVIDIA MIG (Multi-Instance GPU) for H100: partition one H100 into 7 isolated instances. vLLM supports multi-model serving with shared KV cache pool.

---

### Q17. What is model sharding, and when would you use it?
Split model parameters across multiple GPUs (tensor or pipeline parallelism). Use when model is too large for one GPU's VRAM. Example: Llama 3 405B (810GB in FP16) requires sharding across 10+ A100s. Tools: Megatron-LM, DeepSpeed, vLLM with tensor parallelism.

---

### Q18. Request queuing and priority scheduling for AI services
Priority queue (Redis Sorted Set by priority + timestamp): P0 (real-time user requests) → P1 (enterprise users) → P2 (batch/analytics). Worker pool consumes from highest-priority first. Max wait time per priority level. Dead letter queue for requests that wait too long.

---

### Q19. Cost trade-offs: self-hosted vs API-based AI inference
API: zero infra overhead, latest models, pay-per-use (expensive at scale). Self-hosted: high upfront GPU cost, full control, 10-100× cheaper at scale (>1M tokens/day). Break-even: typically 1-2M tokens/day for a 7B model (A100 server amortised over 3 years vs API costs).

---

### Q20. Cold start latency for serverless AI deployments
Model loading takes 30-120 seconds for large models (downloading weights, loading to GPU). Mitigations: pre-warming (keep minimum instances warm), model caching on local SSD (not re-downloading), provisioned concurrency (AWS Lambda), lighter weight models for serverless.

---

### Q21. Model caching to reduce redundant computations
KV prefix cache: cache KV for shared system prompt prefix (vLLM's KV sharing). Response cache: identical inputs → cached outputs (Redis). Embedding cache: cache query embeddings (same query asked repeatedly). Semantic cache: similar queries get similar cached responses.

---

### Q22. Synchronous vs asynchronous inference — when to use each
**Synchronous:** user waits for response. Use for: interactive chat, real-time QA. Max acceptable latency: 5-30s.
**Asynchronous:** user gets a job ID; result polled or notified via webhook. Use for: document processing, bulk generation, multi-step research. Enables processing without blocking user.

---

### Q23. FSDP vs DeepSpeed ZeRO
Both shard model state across GPUs to enable training of large models on limited VRAM.
**FSDP** (PyTorch native): shards parameters + gradients + optimiser states. Simpler integration. Good for single-node or small multi-node.
**DeepSpeed ZeRO Stage 3**: shards everything including activations. More aggressive memory savings. Better for very large models. Also supports CPU offloading.

---

### Q24. Monitoring and profiling LLM inference in production
Key metrics: TTFT (time to first token), ITL (inter-token latency), TGS (tokens generated per second), GPU MFU (model FLOP utilisation), KV cache utilisation, request queue depth. Tools: vLLM metrics endpoint (Prometheus), NVIDIA DCGM for GPU metrics, custom middleware timing.

---

### Q25. Model routing at infrastructure level — route by complexity and cost
Classify query complexity (fast LLM classifier or heuristic: token count, keywords) → route simple queries to cheap fast model (Llama 3 8B via Groq), complex to frontier model (GPT-4o). Latency routing: if Groq overloaded, route to OpenAI. Cost routing: track budget; downgrade model when 80% consumed.

---

### Q26-27: Summary
**Q26. LLM Routing:** semantic routing + model routing combined. Route by topic (domain-specific model), by complexity (cheap vs expensive model), by SLA (real-time vs batch), by availability (circuit-breaker-based fallback).

**Q27. GPU cost optimisation:** Spot/preemptible instances (60-80% cheaper); auto-scale to zero during off-hours; use smaller quantised models; batch async requests; profile and eliminate wasted compute.
